<a href="https://colab.research.google.com/github/mdrs-thiago/mdrs-thiago.github.io/blob/main/topicos_a/2027/notebooks/lab03_deteccao_segmentacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Lab 3 — Detecção ou segmentação
**Tópicos Especiais A · Deep Learning · UERJ 2027.1** · Aula 12 · Bloco 3 — Visão Aplicada

**Objetivo:** treinar e avaliar um modelo de detecção (YOLO) **ou** de segmentação (U-Net), com análise de falhas.

| | |
|---|---|
| ✅ **Checkpoint (em sala)** | 60 XP: veja as marcações `✅ CHECKPOINT` |
| 📦 **Entrega (7 dias)** | 180 XP: métrica antes/depois, predições × anotações, 5 falhas comentadas + proposta do projeto final. |

> Preencha apenas os trechos marcados com `# TODO`. Antes de começar: **Arquivo → Salvar uma cópia no Drive** e, se indicado, **Ambiente de execução → Alterar tipo → T4 GPU**.

## Escolha sua trilha
- **Trilha D (detecção)**: seções D1–D5
- **Trilha S (segmentação)**: seções S1–S5

Rode só a trilha escolhida. Ligue a GPU.

In [ ]:
import random, numpy as np, torch
def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| torch', torch.__version__)

---
# 🅳 Trilha D: Detecção com YOLO

In [ ]:
!pip -q install ultralytics

### D1. Dataset
Baixe um dataset pequeno com **classes fora do COCO** no formato YOLOv8/YOLO11 (ex.: Roboflow Universe → *Download dataset → YOLOv8*),
ou anote ≥ 100 imagens suas. A estrutura esperada:
```
dataset/
  data.yaml        # caminhos train/val + names
  train/images, train/labels
  valid/images, valid/labels
```

In [ ]:
DATA_YAML = 'dataset/data.yaml'   # TODO: ajuste o caminho
import yaml; print(yaml.safe_load(open(DATA_YAML)))

### D2. Visualize as anotações (bug nº 1 em detecção!)

In [ ]:
import glob, cv2, matplotlib.pyplot as plt, os
imgs = sorted(glob.glob('dataset/train/images/*'))[:4]
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for ax, p in zip(axs, imgs):
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB); H, W = im.shape[:2]
    lab = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    for line in open(lab) if os.path.exists(lab) else []:
        c, xc, yc, w, h = map(float, line.split()[:5])
        # TODO: converta (xc, yc, w, h) normalizados para pixels xyxy e desenhe com cv2.rectangle
        ...
    ax.imshow(im); ax.axis('off')

### D3. ✅ CHECKPOINT: métrica de referência do modelo sem treino

In [ ]:
from ultralytics import YOLO
base = YOLO('yolo11n.pt')
m0 = base.val(data=DATA_YAML)   # classes fora do COCO → mAP ~0 (por quê?)
print('mAP50-95 antes:', m0.box.map)

### D4. Fine-tuning

In [ ]:
model = YOLO('yolo11n.pt')
# TODO: treine por ~30 épocas, imgsz=640, seed=42
model.train(...)
m1 = model.val(data=DATA_YAML)
print('mAP50:', m1.box.map50, '| mAP50-95:', m1.box.map)

### D5. Predições e falhas

In [ ]:
val_imgs = sorted(glob.glob('dataset/valid/images/*'))[:6]
res = model(val_imgs, conf=0.25)
fig, axs = plt.subplots(2, 3, figsize=(15, 8))
for ax, r in zip(axs.flat, res):
    ax.imshow(r.plot()[..., ::-1]); ax.axis('off')

---
# 🆂 Trilha S: Segmentação com U-Net

In [ ]:
!pip -q install segmentation-models-pytorch

### S1. Dataset (Oxford-IIIT Pets, máscaras trimap: 1 = animal, 2 = fundo, 3 = borda)

In [ ]:
import torch.nn as nn, torch.nn.functional as F, matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, tv_tensors
from torchvision.transforms import v2

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
# Transformações CONJUNTAS: a mesma operação geométrica na imagem e na máscara
train_tfm = v2.Compose([v2.RandomResizedCrop(256, scale=(0.6, 1.0)), v2.RandomHorizontalFlip(),
                        v2.ToImage(), v2.ToDtype({tv_tensors.Image: torch.float32, 'others': None}, scale=True),
                        v2.Normalize(MEAN, STD)])
val_tfm = v2.Compose([v2.Resize((256, 256)), v2.ToImage(),
                      v2.ToDtype({tv_tensors.Image: torch.float32, 'others': None}, scale=True), v2.Normalize(MEAN, STD)])

class PetsSeg(torch.utils.data.Dataset):
    def __init__(self, split, tfm):
        self.ds = datasets.OxfordIIITPet('data', split=split, target_types='segmentation', download=True)
        self.tfm = tfm
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        img, mask = self.ds[i]
        img, mask = self.tfm(tv_tensors.Image(v2.functional.pil_to_tensor(img)), tv_tensors.Mask(v2.functional.pil_to_tensor(mask)))
        return img, (mask[0].long() - 1)   # classes 0 (animal), 1 (fundo), 2 (borda)

full_tr, full_val = PetsSeg('trainval', train_tfm), PetsSeg('trainval', val_tfm)
idx = torch.randperm(len(full_tr), generator=torch.Generator().manual_seed(42)).tolist()
train_ds, val_ds = Subset(full_tr, idx[500:]), Subset(full_val, idx[:500])
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, num_workers=2)

### S2. Visualize imagem + máscara (sanidade)

In [ ]:
x, y = val_ds[0]
fig, (a, b) = plt.subplots(1, 2, figsize=(8, 4))
a.imshow((x.permute(1, 2, 0) * torch.tensor(STD) + torch.tensor(MEAN)).clamp(0, 1)); b.imshow(y)
print(x.shape, y.shape, y.unique())

### S3. Métricas, baseline e ✅ CHECKPOINT

In [ ]:
@torch.no_grad()
def iou_dice(model, loader, n_cls=3):
    inter, union, psum, tsum = [torch.zeros(n_cls) for _ in range(4)]
    model.eval()
    for xb, yb in loader:
        pred = model(xb.to(device)).argmax(1).cpu()
        for c in range(n_cls):
            p, t = pred == c, yb == c
            inter[c] += (p & t).sum(); union[c] += (p | t).sum(); psum[c] += p.sum(); tsum[c] += t.sum()
    iou = inter / union.clamp(min=1); dice = 2 * inter / (psum + tsum).clamp(min=1)
    return iou, dice

class AllBackground(nn.Module):   # baseline: prevê "fundo" em todo pixel
    def forward(self, x):
        out = torch.zeros(x.shape[0], 3, *x.shape[2:], device=x.device); out[:, 1] = 1; return out
print('baseline IoU por classe (animal, fundo, borda):', iou_dice(AllBackground(), val_loader)[0])

### S4. U-Net e treino (perda CE + Dice)

In [ ]:
import segmentation_models_pytorch as smp
model = smp.Unet('resnet34', encoder_weights='imagenet', classes=3).to(device)

def dice_loss(logits, target, eps=1e-6):
    p = logits.softmax(1); t = F.one_hot(target, 3).permute(0, 3, 1, 2).float()
    inter = (p * t).sum((2, 3)); denom = p.sum((2, 3)) + t.sum((2, 3))
    return 1 - ((2 * inter + eps) / (denom + eps)).mean()

opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
for ep in range(8):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        # TODO: loss = entropia cruzada + dice_loss; depois zero_grad, backward, step
        loss = ...
    iou, dice = iou_dice(model, val_loader)
    print(f'ep {ep+1} | IoU {iou.numpy().round(3)} | Dice {dice.numpy().round(3)}')

### S5. Predições e falhas

In [ ]:
model.eval(); xb, yb = next(iter(val_loader))
with torch.no_grad(): pred = model(xb.to(device)).argmax(1).cpu()
fig, axs = plt.subplots(3, 6, figsize=(18, 9))
for k in range(6):
    axs[0, k].imshow((xb[k].permute(1, 2, 0) * torch.tensor(STD) + torch.tensor(MEAN)).clamp(0, 1))
    axs[1, k].imshow(yb[k]); axs[2, k].imshow(pred[k])
    for r in range(3): axs[r, k].axis('off')
axs[0, 0].set_title('imagem'); axs[1, 0].set_title('anotação'); axs[2, 0].set_title('predição')

---
## 📦 Análise (ambas as trilhas)
1. Métrica principal **antes** e **depois** do treino: ...
2. **5 casos de falha** (imagem + hipótese: oclusão? escala? classe parecida? borda fina? iluminação?):
   1. ...
3. O que você mudaria com mais 1 semana? ...

## 📄 Lembrete: proposta do projeto final (mesmo prazo)
1 página: problema · dados (fonte, tamanho, licença) · métrica · baseline · blocos usados (≥ 2) · dupla ou individual.